
### Workflow Summary

This notebook outlines a data engineering workflow using Delta Lake. It covers loading raw CSV data into DataFrames, writing and reading data with Delta Lake, and structuring data into Silver and Gold schemas. The Silver layer contains cleaned and enriched data, while the Gold layer models data for analytics and reporting using a star schema.

## Data Load

This section describes loading ingested CSV files into DataFrames, preparing them for export to Delta tables. The process involves reading raw CSV data, applying necessary transformations, and structuring the DataFrames for efficient Delta Lake storage.

In [0]:
# These are the file paths where our data is stored
payments = "/Volumes/proj4/default/proj4data/prj4Data/payments.csv"
riders   = "/Volumes/proj4/default/proj4data/prj4Data/riders.csv"
stations = "/Volumes/proj4/default/proj4data/prj4Data/stations.csv"
trips    = "/Volumes/proj4/default/proj4data/prj4Data/trips.csv"

# Column lists for each table
table_info = {
    "df_payments": (payments, ["payment_id", "date", "amount", "rider_id"]),
    "df_riders":   (riders,   ["rider_id", "first_name", "last_name", "address", "birthday", "account_start_date", "account_end_date", "is_member"]),
    "df_stations": (stations, ["station_id", "name", "latitude", "longitude"]),
    "df_trips":    (trips,    ["trip_id", "rideable_type", "start_at", "ended_at", "start_station_id", "end_station_id", "rider_id"])
}

# Function to read CSV and display first 5 rows
def read_and_display(name, file_path, columns):
    df = spark.read.format("csv")\
        .option("inferSchema", "false")\
        .option("header", "true")\
        .option("sep", ",")\
        .load(file_path)\
        .toDF(*columns)
    print(f"{'*' * 18}{' ' * 2}{name}{' ' * 2}{'*' * 18}")
    df.printSchema()
    display(df.limit(5))
    return df

# Read and display all tables
dfs = {name: read_and_display(name, path, cols) for name, (path, cols) in table_info.items()}

# Assign DataFrames to variables for later use
df_payments = dfs["df_payments"]
df_riders = dfs["df_riders"]
df_stations = dfs["df_stations"]
df_trips = dfs["df_trips"]

******************  df_payments  ******************
root
 |-- payment_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- rider_id: string (nullable = true)



payment_id,date,amount,rider_id
2,2019-06-01,9.0,1000
3,2019-07-01,9.0,1000
4,2019-08-01,9.0,1000
5,2019-09-01,9.0,1000
6,2019-10-01,9.0,1000


******************  df_riders  ******************
root
 |-- rider_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- birthday: string (nullable = true)
 |-- account_start_date: string (nullable = true)
 |-- account_end_date: string (nullable = true)
 |-- is_member: string (nullable = true)



rider_id,first_name,last_name,address,birthday,account_start_date,account_end_date,is_member
1001,Jennifer,Smith,397 Diana Ferry,1976-08-10,2019-11-01,2020-09-01,True
1002,Karen,Smith,644 Brittany Row Apt. 097,1998-08-10,2022-02-04,null,True
1003,Bryan,Roberts,996 Dickerson Turnpike,1999-03-29,2019-08-26,null,False
1004,Jesse,Middleton,7009 Nathan Expressway,1969-04-11,2019-09-14,null,True
1005,Christine,Rodriguez,224 Washington Mills Apt. 467,1974-08-27,2020-03-24,null,False


******************  df_stations  ******************
root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)



station_id,name,latitude,longitude
KA1503000012,Clark St & Lake St,41.88579466666667,-87.63110066666668
637,Wood St & Chicago Ave,41.895634,-87.672069
13216,State St & 33rd St,41.8347335,-87.6258275
18003,Fairbanks St & Superior St,41.89580766666667,-87.62025316666669
KP1705001026,LaSalle Dr & Huron St,41.894877,-87.632326


******************  df_trips  ******************
root
 |-- trip_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- start_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- rider_id: string (nullable = true)



trip_id,rideable_type,start_at,ended_at,start_station_id,end_station_id,rider_id
0FEFDE2603568365,classic_bike,2021-02-14 17:52:38,2021-02-14 18:12:09,525,16806,47854
E6159D746B2DBB91,electric_bike,2021-02-09 19:10:18,2021-02-09 19:19:10,KA1503000012,TA1305000029,70870
B32D3199F1C2E75B,classic_bike,2021-02-02 17:49:41,2021-02-02 17:54:06,637,TA1305000034,58974
83E463F23575F4BF,electric_bike,2021-02-23 15:07:23,2021-02-23 15:22:37,13216,TA1309000055,39608
BDAA7E3494E8D545,electric_bike,2021-02-24 15:43:33,2021-02-24 15:49:05,18003,KP1705001026,36267


## Delta Lake Data Write and Read Example

This section demonstrates how to write data to a Delta folder and read it back, showcasing Delta Lake's capabilities for efficient data storage and retrieval.

In [0]:
# Function to write all DataFrames in the dictionary to Delta format at specified paths
def write_csv_to_delta_all(dfs):
    for name, df in dfs.items():
        # Construct Delta path for each DataFrame
        delta_path = f"/Volumes/proj4/default/proj4data/prj4Data/{name}_delta"
        # Write DataFrame to Delta format, overwriting existing data
        df.write.format("delta").mode("overwrite").save(delta_path)

# Write all loaded DataFrames to Delta format
write_csv_to_delta_all(dfs)

In [0]:
# Function to reload DataFrames from Delta paths and overwrite existing variables
def reload_dfs_from_delta(names):
    delta_paths = {name: f"/Volumes/proj4/default/proj4data/prj4Data/{name}_delta" for name in names}
    return {name: spark.read.format("delta").load(path) for name, path in delta_paths.items()}

# Reload and overwrite DataFrames
dfs = reload_dfs_from_delta(["df_payments", "df_riders", "df_stations", "df_trips"])
df_payments = dfs["df_payments"]
df_riders = dfs["df_riders"]
df_stations = dfs["df_stations"]
df_trips = dfs["df_trips"]

## Silver Schema  

This section describes creating the Silver schema and storing files in the Silver layer.
The Silver layer typically contains cleaned and enriched data, structured for analytics.

In [0]:
%sql
create schema if not exists silver;

In [0]:
# This function saves our tables so we can use them later
def write_to_delta(df, table_name):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

# We save each table with a name so we can find them easily
write_to_delta(df_payments, "silver.payments")
write_to_delta(df_riders, "silver.riders")
write_to_delta(df_stations, "silver.stations")
write_to_delta(df_trips, "silver.trips")

##Gold Schema  

This section describes creating the Gold schema, which involves modeling data using a star schema with dimension and fact tables for analytics and reporting.
The Silver layer typically contains cleaned and enriched data, structured for analytics.

In [0]:
# NOTE: This code generates the dim_time table, including both start and end timestamps from trips.
# The resulting dim_time can be linked to any timestamp column in trips (e.g., start_at or ended_at).

from pyspark.sql import functions as f

# Combine start_at and ended_at into one timestamp column, get unique timestamps
unique_timestamps = df_trips.select(
    f.col("start_at").cast("timestamp").alias("timestamp")
).union(
    df_trips.select(f.col("ended_at").cast("timestamp").alias("timestamp"))
).distinct()

# Build dim_time from unique timestamps, include timestamp_id
dim_time = unique_timestamps \
    .withColumn("timestamp_id", f.date_format("timestamp", "yyyyMMddHHmmss")) \
    .withColumn("date", f.to_date("timestamp")) \
    .withColumn("hour", f.hour("timestamp")) \
    .withColumn("minute", f.minute("timestamp")) \
    .withColumn("second", f.second("timestamp")) \
    .withColumn("day_of_week", f.dayofweek("timestamp")) \
    .withColumn("day_of_month", f.dayofmonth("timestamp")) \
    .withColumn("week_of_year", f.weekofyear("timestamp")) \
    .withColumn("year", f.year("timestamp")) \
    .withColumn("quarter", f.quarter("timestamp")) \
    .withColumn("month", f.month("timestamp")) \
    .dropDuplicates(["timestamp_id"])

In [0]:
%sql
create schema if not exists gold;

In [0]:
# Function to save DataFrame to gold schema with dim/fact in table name
def save_gold_table(df, base_name, table_type):
    table_name = f"gold.{table_type}_{base_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

# Save tables using the function
save_gold_table(df_trips, "trip", "fact")
save_gold_table(df_payments, "payment", "fact")
save_gold_table(df_stations, "stations", "dim")
save_gold_table(df_riders, "riders", "dim")
save_gold_table(dim_time, "time", "dim")